# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [32]:
# Load the libraries as required.
import os
import sys
sys.path.append(os.getenv('SRC_DIR'))
import pandas as pd
import numpy as np
import os

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn import linear_model
from sklearn.ensemble import GradientBoostingRegressor

In [33]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [34]:
X = fires_dt.drop(columns='area')

In [35]:
Y = fires_dt['area']

In [36]:
#scoring = ['neg_log_loss', 'roc_auc', 'f1', 'accuracy', 'precision', 'recall','neg_mean_squared_error']
scoring = ['neg_mean_absolute_error','neg_mean_squared_error']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [37]:
preproc1 = ColumnTransformer(
    transformers=[
        ('numeric_transfomer', StandardScaler(), ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain'] ),
        ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist'), ['month', 'day']), 
    ], remainder='passthrough'
)

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [38]:
from sklearn.preprocessing import FunctionTransformer
#from sklearn.preprocessing import PowerTransformer
pt = FunctionTransformer()

preproc2 = ColumnTransformer(
    transformers=[
        ('numeric_transfomer', StandardScaler(), ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain'] ),
        ('Power_transfomer', pt, ['temp'] ),
        ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist'), ['month', 'day']), 
    ], remainder='passthrough'
)


## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [39]:
# Pipeline A = preproc1 + baseline
pipe_knr = Pipeline([
    ('preprocess', preproc1),
    ('knr', KNeighborsRegressor())
    #('knr', linear_model.Ridge())
])
pipe_knr




Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('knr', KNeighborsRegressor())])

In [40]:
# Pipeline B = preproc2 + baseline
pipe_knr2 = Pipeline([
    ('preprocess2', preproc2),
    ('knr2', KNeighborsRegressor())
])
pipe_knr2

Pipeline(steps=[('preprocess2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('Power_transfomer',
                                                  FunctionTransformer(),
                                                  ['temp']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('knr2', KNeighborsRegressor())])

In [41]:
# Pipeline C = preproc1 + advanced model

pipe_knr3 = Pipeline([
    ('preprocess3', preproc1),
    ('knr3', GradientBoostingRegressor())
])
pipe_knr3

Pipeline(steps=[('preprocess3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('knr3', GradientBoostingRegressor())])

In [42]:
# Pipeline D = preproc2 + advanced model

pipe_knr4 = Pipeline([
    ('preprocess4', preproc2),
    ('knr4', GradientBoostingRegressor())
    ])
pipe_knr4

Pipeline(steps=[('preprocess4',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('Power_transfomer',
                                                  FunctionTransformer(),
                                                  ['temp']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('knr4', GradientBoostingRegressor())])

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [43]:
param_grid1 = {
    'knr__n_neighbors': [3, 5, 7,9],
    'knr__p': [1,2]
    }

grid_cv1 = GridSearchCV(
    estimator=pipe_knr, 
    param_grid=param_grid1, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_mean_squared_error")

grid_cv1.fit(X_train, Y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numeric_transfomer',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('onehot',
                                                                         OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('knr', KNeighborsRegressor())]),
             param_grid={'knr__n_neighbors': [3, 5, 7, 9], 'knr__p': [1, 2]},
             refit='neg_mean_squared_error',
             scoring=['neg_mean_absolute_error', 'neg_mean_squared_error'])

In [44]:
res1 = grid_cv1.cv_results_
res1 = pd.DataFrame(res1)
res1

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr__n_neighbors,param_knr__p,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
0,0.032559,0.044015,0.057778,0.087128,3,1,"{'knr__n_neighbors': 3, 'knr__p': 1}",-16.449157,-15.533936,-17.184779,...,3.889249,5,-1873.187968,-1218.214595,-1451.674325,-7776.906675,-1446.432067,-2753.283126,2520.707901,7
1,0.006646,0.002986,0.014226,0.007093,3,2,"{'knr__n_neighbors': 3, 'knr__p': 2}",-21.212851,-16.434137,-17.974498,...,3.747099,7,-2813.326232,-1608.239108,-2102.398837,-7812.569202,-1035.386921,-3074.384060,2440.098164,8
2,0.010128,0.005346,0.009264,0.003672,5,1,"{'knr__n_neighbors': 5, 'knr__p': 1}",-20.471060,-18.261687,-16.213084,...,4.019720,8,-2377.314564,-1346.299998,-1161.860454,-7756.846647,-1021.164247,-2732.697182,2556.847466,6
3,0.009177,0.003433,0.009780,0.003992,5,2,"{'knr__n_neighbors': 5, 'knr__p': 2}",-17.999157,-17.034410,-16.389687,...,3.851314,4,-1993.330721,-1291.160163,-1171.387382,-7689.458221,-645.651546,-2558.197607,2601.359991,4
4,0.007183,0.005298,0.011763,0.004512,7,1,"{'knr__n_neighbors': 7, 'knr__p': 1}",-20.923804,-17.772117,-17.131962,...,4.227390,6,-2477.958047,-981.888595,-1023.938486,-7603.609229,-775.558697,-2572.590611,2587.588981,5
5,0.007647,0.003635,0.011037,0.003394,7,2,"{'knr__n_neighbors': 7, 'knr__p': 2}",-18.251222,-15.238864,-15.954768,...,4.018582,2,-1910.257821,-772.865721,-984.583811,-7401.428245,-571.227434,-2328.072606,2577.737077,2
6,0.011277,0.004889,0.024484,0.022473,9,1,"{'knr__n_neighbors': 9, 'knr__p': 1}",-18.646345,-15.658527,-16.957323,...,3.998131,3,-1901.599844,-674.770463,-939.734348,-7471.110576,-684.242932,-2334.291633,2607.511928,3
7,0.007243,0.003964,0.009385,0.004735,9,2,"{'knr__n_neighbors': 9, 'knr__p': 2}",-17.451071,-15.100161,-15.519344,...,3.896534,1,-1729.479837,-641.062780,-884.707057,-7357.961542,-656.778963,-2253.998036,2582.757809,1


In [45]:
res1 = grid_cv1.cv_results_
res1 = pd.DataFrame(res1)
##res1
res1.columns

Index(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_knr__n_neighbors', 'param_knr__p', 'params',
       'split0_test_neg_mean_absolute_error',
       'split1_test_neg_mean_absolute_error',
       'split2_test_neg_mean_absolute_error',
       'split3_test_neg_mean_absolute_error',
       'split4_test_neg_mean_absolute_error',
       'mean_test_neg_mean_absolute_error', 'std_test_neg_mean_absolute_error',
       'rank_test_neg_mean_absolute_error',
       'split0_test_neg_mean_squared_error',
       'split1_test_neg_mean_squared_error',
       'split2_test_neg_mean_squared_error',
       'split3_test_neg_mean_squared_error',
       'split4_test_neg_mean_squared_error',
       'mean_test_neg_mean_squared_error', 'std_test_neg_mean_squared_error',
       'rank_test_neg_mean_squared_error'],
      dtype='object')

In [46]:
res1 = grid_cv1.cv_results_
res1 = pd.DataFrame(res1)
##res1
res1.columns

res1[['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_knr__n_neighbors', 'param_knr__p', 'params',
       'split0_test_neg_mean_absolute_error',
       'split1_test_neg_mean_absolute_error',
       'split2_test_neg_mean_absolute_error',
       'split3_test_neg_mean_absolute_error',
       'split4_test_neg_mean_absolute_error',
       'mean_test_neg_mean_absolute_error', 'std_test_neg_mean_absolute_error',
       'rank_test_neg_mean_absolute_error',
       'split0_test_neg_mean_squared_error',
       'split1_test_neg_mean_squared_error',
       'split2_test_neg_mean_squared_error',
       'split3_test_neg_mean_squared_error',
       'split4_test_neg_mean_squared_error',
       'mean_test_neg_mean_squared_error', 'std_test_neg_mean_squared_error',
       'rank_test_neg_mean_squared_error']].sort_values('rank_test_neg_mean_squared_error')

#Note that best param {knr__n_neighbors: 9,knr__p:2}, Avg mean = -2253.9


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr__n_neighbors,param_knr__p,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
7,0.007243,0.003964,0.009385,0.004735,9,2,"{'knr__n_neighbors': 9, 'knr__p': 2}",-17.451071,-15.100161,-15.519344,...,3.896534,1,-1729.479837,-641.062780,-884.707057,-7357.961542,-656.778963,-2253.998036,2582.757809,1
5,0.007647,0.003635,0.011037,0.003394,7,2,"{'knr__n_neighbors': 7, 'knr__p': 2}",-18.251222,-15.238864,-15.954768,...,4.018582,2,-1910.257821,-772.865721,-984.583811,-7401.428245,-571.227434,-2328.072606,2577.737077,2
6,0.011277,0.004889,0.024484,0.022473,9,1,"{'knr__n_neighbors': 9, 'knr__p': 1}",-18.646345,-15.658527,-16.957323,...,3.998131,3,-1901.599844,-674.770463,-939.734348,-7471.110576,-684.242932,-2334.291633,2607.511928,3
3,0.009177,0.003433,0.009780,0.003992,5,2,"{'knr__n_neighbors': 5, 'knr__p': 2}",-17.999157,-17.034410,-16.389687,...,3.851314,4,-1993.330721,-1291.160163,-1171.387382,-7689.458221,-645.651546,-2558.197607,2601.359991,4
4,0.007183,0.005298,0.011763,0.004512,7,1,"{'knr__n_neighbors': 7, 'knr__p': 1}",-20.923804,-17.772117,-17.131962,...,4.227390,6,-2477.958047,-981.888595,-1023.938486,-7603.609229,-775.558697,-2572.590611,2587.588981,5
2,0.010128,0.005346,0.009264,0.003672,5,1,"{'knr__n_neighbors': 5, 'knr__p': 1}",-20.471060,-18.261687,-16.213084,...,4.019720,8,-2377.314564,-1346.299998,-1161.860454,-7756.846647,-1021.164247,-2732.697182,2556.847466,6
0,0.032559,0.044015,0.057778,0.087128,3,1,"{'knr__n_neighbors': 3, 'knr__p': 1}",-16.449157,-15.533936,-17.184779,...,3.889249,5,-1873.187968,-1218.214595,-1451.674325,-7776.906675,-1446.432067,-2753.283126,2520.707901,7
1,0.006646,0.002986,0.014226,0.007093,3,2,"{'knr__n_neighbors': 3, 'knr__p': 2}",-21.212851,-16.434137,-17.974498,...,3.747099,7,-2813.326232,-1608.239108,-2102.398837,-7812.569202,-1035.386921,-3074.384060,2440.098164,8


In [47]:
Best_param1 = grid_cv1.best_params_
#Best_param1 = pd.DataFrame(Best_param1)
print(Best_param1)
print(grid_cv1.best_estimator_)

{'knr__n_neighbors': 9, 'knr__p': 2}
Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('knr', KNeighborsRegressor(n_neighbors=9))])


In [48]:
param_grid2 = {
    'knr2__n_neighbors': [3, 5, 7,9],
    'knr2__p': [1,2]
    }


grid_cv2 = GridSearchCV(
    estimator=pipe_knr2, 
    param_grid=param_grid2, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_mean_squared_error")

grid_cv2.fit(X_train, Y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numeric_transfomer',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('Power_transfomer',
                                                                         FunctionTransformer(),
                                                                         ['temp']),
                                                                        ('onehot',
                                                                         OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('knr2', KNeighborsRegressor())]),
             param_grid={'knr2__n_neighbors': [3, 5, 7, 9], 'knr2__p': [1, 2]},
             refit='neg_mean_squared_error',
             scoring=['neg_mean_absolute_error', 'neg_mean_squared_error'])

In [49]:
res2 = grid_cv2.cv_results_
res2 = pd.DataFrame(res2)
res2

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr2__n_neighbors,param_knr2__p,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
0,0.012283,0.007771,0.012135,0.004144,3,1,"{'knr2__n_neighbors': 3, 'knr2__p': 1}",-17.627349,-19.975984,-19.183454,...,3.566867,8,-1939.175732,-2020.643700,-1741.021132,-7663.872051,-978.036767,-2868.549876,2425.825914,8
1,0.009234,0.004418,0.012496,0.003317,3,2,"{'knr2__n_neighbors': 3, 'knr2__p': 2}",-14.987108,-20.503936,-13.972450,...,3.949761,4,-1833.918297,-2112.196932,-856.201301,-7236.719594,-1079.006904,-2623.608606,2352.677992,6
2,0.009508,0.003972,0.010077,0.004642,5,1,"{'knr2__n_neighbors': 5, 'knr2__p': 1}",-19.651831,-18.097277,-18.797783,...,3.516038,7,-2363.566108,-1200.156069,-1393.447609,-7513.018880,-700.535121,-2634.144757,2498.444504,7
3,0.010207,0.004193,0.012710,0.004129,5,2,"{'knr2__n_neighbors': 5, 'knr2__p': 2}",-16.396289,-16.562771,-14.194145,...,3.114068,3,-1949.759575,-966.609921,-847.609792,-7318.201060,-1057.548943,-2427.945858,2476.053738,4
4,0.009761,0.006869,0.011757,0.003481,7,1,"{'knr2__n_neighbors': 7, 'knr2__p': 1}",-18.786902,-16.471377,-18.522444,...,3.747471,6,-2111.917956,-749.030636,-1331.544077,-7317.116646,-849.013322,-2471.724527,2470.148292,5
5,0.009742,0.003247,0.011582,0.002518,7,2,"{'knr2__n_neighbors': 7, 'knr2__p': 2}",-14.828640,-14.400258,-16.855938,...,3.494109,2,-1555.695932,-650.224353,-1157.582064,-6942.610238,-821.009093,-2225.424336,2378.835986,2
6,0.009762,0.002991,0.010149,0.003091,9,1,"{'knr2__n_neighbors': 9, 'knr2__p': 1}",-17.043467,-17.011486,-18.010067,...,3.755056,5,-1607.519929,-900.159146,-1138.703982,-7017.940723,-745.236204,-2281.911997,2385.888126,3
7,0.013099,0.004436,0.011398,0.003985,9,2,"{'knr2__n_neighbors': 9, 'knr2__p': 2}",-14.929786,-14.416827,-16.194659,...,3.531032,1,-1539.203920,-633.054250,-998.444498,-6948.007445,-761.715022,-2176.085027,2406.037606,1


In [50]:
res2.columns

Index(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_knr2__n_neighbors', 'param_knr2__p', 'params',
       'split0_test_neg_mean_absolute_error',
       'split1_test_neg_mean_absolute_error',
       'split2_test_neg_mean_absolute_error',
       'split3_test_neg_mean_absolute_error',
       'split4_test_neg_mean_absolute_error',
       'mean_test_neg_mean_absolute_error', 'std_test_neg_mean_absolute_error',
       'rank_test_neg_mean_absolute_error',
       'split0_test_neg_mean_squared_error',
       'split1_test_neg_mean_squared_error',
       'split2_test_neg_mean_squared_error',
       'split3_test_neg_mean_squared_error',
       'split4_test_neg_mean_squared_error',
       'mean_test_neg_mean_squared_error', 'std_test_neg_mean_squared_error',
       'rank_test_neg_mean_squared_error'],
      dtype='object')

In [ ]:
res2 = grid_cv2.cv_results_
res2 = pd.DataFrame(res2)
#res2
res2.columns

res2[['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_knr2__n_neighbors', 'param_knr2__p', 'params',
       'split0_test_neg_mean_absolute_error',
       'split1_test_neg_mean_absolute_error',
       'split2_test_neg_mean_absolute_error',
       'split3_test_neg_mean_absolute_error',
       'split4_test_neg_mean_absolute_error',
       'mean_test_neg_mean_absolute_error', 'std_test_neg_mean_absolute_error',
       'rank_test_neg_mean_absolute_error',
       'split0_test_neg_mean_squared_error',
       'split1_test_neg_mean_squared_error',
       'split2_test_neg_mean_squared_error',
       'split3_test_neg_mean_squared_error',
       'split4_test_neg_mean_squared_error',
       'mean_test_neg_mean_squared_error', 'std_test_neg_mean_squared_error',
       'rank_test_neg_mean_squared_error']].sort_values('rank_test_neg_mean_squared_error')


#Note that best param {knr2__n_neighbors: 9,knr__p:2}, avg Mean -2176

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr2__n_neighbors,param_knr2__p,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
7,0.013099,0.004436,0.011398,0.003985,9,2,"{'knr2__n_neighbors': 9, 'knr2__p': 2}",-14.929786,-14.416827,-16.194659,...,3.531032,1,-1539.203920,-633.054250,-998.444498,-6948.007445,-761.715022,-2176.085027,2406.037606,1
5,0.009742,0.003247,0.011582,0.002518,7,2,"{'knr2__n_neighbors': 7, 'knr2__p': 2}",-14.828640,-14.400258,-16.855938,...,3.494109,2,-1555.695932,-650.224353,-1157.582064,-6942.610238,-821.009093,-2225.424336,2378.835986,2
6,0.009762,0.002991,0.010149,0.003091,9,1,"{'knr2__n_neighbors': 9, 'knr2__p': 1}",-17.043467,-17.011486,-18.010067,...,3.755056,5,-1607.519929,-900.159146,-1138.703982,-7017.940723,-745.236204,-2281.911997,2385.888126,3
3,0.010207,0.004193,0.012710,0.004129,5,2,"{'knr2__n_neighbors': 5, 'knr2__p': 2}",-16.396289,-16.562771,-14.194145,...,3.114068,3,-1949.759575,-966.609921,-847.609792,-7318.201060,-1057.548943,-2427.945858,2476.053738,4
4,0.009761,0.006869,0.011757,0.003481,7,1,"{'knr2__n_neighbors': 7, 'knr2__p': 1}",-18.786902,-16.471377,-18.522444,...,3.747471,6,-2111.917956,-749.030636,-1331.544077,-7317.116646,-849.013322,-2471.724527,2470.148292,5
1,0.009234,0.004418,0.012496,0.003317,3,2,"{'knr2__n_neighbors': 3, 'knr2__p': 2}",-14.987108,-20.503936,-13.972450,...,3.949761,4,-1833.918297,-2112.196932,-856.201301,-7236.719594,-1079.006904,-2623.608606,2352.677992,6
2,0.009508,0.003972,0.010077,0.004642,5,1,"{'knr2__n_neighbors': 5, 'knr2__p': 1}",-19.651831,-18.097277,-18.797783,...,3.516038,7,-2363.566108,-1200.156069,-1393.447609,-7513.018880,-700.535121,-2634.144757,2498.444504,7
0,0.012283,0.007771,0.012135,0.004144,3,1,"{'knr2__n_neighbors': 3, 'knr2__p': 1}",-17.627349,-19.975984,-19.183454,...,3.566867,8,-1939.175732,-2020.643700,-1741.021132,-7663.872051,-978.036767,-2868.549876,2425.825914,8


In [52]:
param_grid3 = {
    'knr3__max_depth': [2, 3, 4,5],
    'knr3__min_samples_leaf': [1,2]
    }

grid_cv3 = GridSearchCV(
    estimator=pipe_knr3, 
    param_grid=param_grid3, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_mean_squared_error")

grid_cv3.fit(X_train, Y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess3',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numeric_transfomer',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('onehot',
                                                                         OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('knr3', GradientBoostingRegressor())]),
             param_grid={'knr3__max_depth': [2, 3, 4, 5],
                         'knr3__min_samples_leaf': [1, 2]},
             refit='neg_mean_squared_error',
             scoring=['neg_mean_absolute_error', 'neg_mean_squared_error'])

In [53]:
res3 = grid_cv3.cv_results_
res3 = pd.DataFrame(res3)
res3
#res3.columns

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr3__max_depth,param_knr3__min_samples_leaf,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
0,0.145133,0.054151,0.007668,0.005128,2,1,"{'knr3__max_depth': 2, 'knr3__min_samples_leaf...",-23.169788,-32.659638,-16.008907,...,6.802928,5,-6477.864861,-6764.323877,-917.611503,-7327.234063,-805.360124,-4458.478886,2949.834423,7
1,0.114187,0.000768,0.006013,0.005445,2,2,"{'knr3__max_depth': 2, 'knr3__min_samples_leaf...",-20.050366,-27.821909,-15.734077,...,5.574965,1,-2442.561391,-3705.846922,-956.238535,-7494.260160,-652.808668,-3050.343135,2477.132535,1
2,0.139007,0.005995,0.008062,0.002725,3,1,"{'knr3__max_depth': 3, 'knr3__min_samples_leaf...",-22.224506,-28.811036,-20.184004,...,5.848120,4,-5199.793478,-5714.654702,-2245.324442,-7669.351393,-688.032158,-4303.431235,2507.243487,6
3,0.137236,0.004200,0.006185,0.001542,3,2,"{'knr3__max_depth': 3, 'knr3__min_samples_leaf...",-22.358969,-26.757947,-19.734021,...,5.134117,2,-3080.803690,-3142.042207,-1292.876994,-7372.310269,-684.729466,-3114.552525,2338.928708,2
4,0.168177,0.009095,0.005241,0.006327,4,1,"{'knr3__max_depth': 4, 'knr3__min_samples_leaf...",-22.489504,-29.914442,-18.568376,...,6.249842,8,-4493.923699,-6458.799766,-1188.550433,-8104.554305,-761.572570,-4201.480155,2874.916088,5
5,0.170931,0.007918,0.004197,0.003911,4,2,"{'knr3__max_depth': 4, 'knr3__min_samples_leaf...",-22.595243,-27.826094,-19.247444,...,4.790444,3,-3457.096266,-2933.352795,-1197.654454,-7190.990932,-846.691238,-3125.157137,2262.063234,3
6,0.189600,0.004531,0.005170,0.003614,5,1,"{'knr3__max_depth': 5, 'knr3__min_samples_leaf...",-23.662884,-28.504918,-18.465169,...,6.126632,6,-6165.942045,-7181.155529,-1055.929361,-7972.272963,-722.747575,-4619.609495,3100.914823,8
7,0.183631,0.007842,0.002138,0.004275,5,2,"{'knr3__max_depth': 5, 'knr3__min_samples_leaf...",-21.949047,-31.797240,-20.753844,...,5.810501,7,-3086.638922,-3639.412137,-1287.614961,-7344.798115,-865.895012,-3244.871829,2301.324853,4


In [54]:
res3.columns

Index(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_knr3__max_depth', 'param_knr3__min_samples_leaf', 'params',
       'split0_test_neg_mean_absolute_error',
       'split1_test_neg_mean_absolute_error',
       'split2_test_neg_mean_absolute_error',
       'split3_test_neg_mean_absolute_error',
       'split4_test_neg_mean_absolute_error',
       'mean_test_neg_mean_absolute_error', 'std_test_neg_mean_absolute_error',
       'rank_test_neg_mean_absolute_error',
       'split0_test_neg_mean_squared_error',
       'split1_test_neg_mean_squared_error',
       'split2_test_neg_mean_squared_error',
       'split3_test_neg_mean_squared_error',
       'split4_test_neg_mean_squared_error',
       'mean_test_neg_mean_squared_error', 'std_test_neg_mean_squared_error',
       'rank_test_neg_mean_squared_error'],
      dtype='object')

In [ ]:
res3[['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_knr3__max_depth', 'param_knr3__min_samples_leaf', 'params',
       'split0_test_neg_mean_absolute_error',
       'split1_test_neg_mean_absolute_error',
       'split2_test_neg_mean_absolute_error',
       'split3_test_neg_mean_absolute_error',
       'split4_test_neg_mean_absolute_error',
       'mean_test_neg_mean_absolute_error', 'std_test_neg_mean_absolute_error',
       'rank_test_neg_mean_absolute_error',
       'split0_test_neg_mean_squared_error',
       'split1_test_neg_mean_squared_error',
       'split2_test_neg_mean_squared_error',
       'split3_test_neg_mean_squared_error',
       'split4_test_neg_mean_squared_error',
       'mean_test_neg_mean_squared_error', 'std_test_neg_mean_squared_error',
       'rank_test_neg_mean_squared_error']].sort_values('rank_test_neg_mean_squared_error')

#Note that best param {'knr3__max_depth': 2, 'knr3__min_samples_leaf': 2}, avg mean = -3050.3

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr3__max_depth,param_knr3__min_samples_leaf,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
1,0.114187,0.000768,0.006013,0.005445,2,2,"{'knr3__max_depth': 2, 'knr3__min_samples_leaf...",-20.050366,-27.821909,-15.734077,...,5.574965,1,-2442.561391,-3705.846922,-956.238535,-7494.260160,-652.808668,-3050.343135,2477.132535,1
3,0.137236,0.004200,0.006185,0.001542,3,2,"{'knr3__max_depth': 3, 'knr3__min_samples_leaf...",-22.358969,-26.757947,-19.734021,...,5.134117,2,-3080.803690,-3142.042207,-1292.876994,-7372.310269,-684.729466,-3114.552525,2338.928708,2
5,0.170931,0.007918,0.004197,0.003911,4,2,"{'knr3__max_depth': 4, 'knr3__min_samples_leaf...",-22.595243,-27.826094,-19.247444,...,4.790444,3,-3457.096266,-2933.352795,-1197.654454,-7190.990932,-846.691238,-3125.157137,2262.063234,3
7,0.183631,0.007842,0.002138,0.004275,5,2,"{'knr3__max_depth': 5, 'knr3__min_samples_leaf...",-21.949047,-31.797240,-20.753844,...,5.810501,7,-3086.638922,-3639.412137,-1287.614961,-7344.798115,-865.895012,-3244.871829,2301.324853,4
4,0.168177,0.009095,0.005241,0.006327,4,1,"{'knr3__max_depth': 4, 'knr3__min_samples_leaf...",-22.489504,-29.914442,-18.568376,...,6.249842,8,-4493.923699,-6458.799766,-1188.550433,-8104.554305,-761.572570,-4201.480155,2874.916088,5
2,0.139007,0.005995,0.008062,0.002725,3,1,"{'knr3__max_depth': 3, 'knr3__min_samples_leaf...",-22.224506,-28.811036,-20.184004,...,5.848120,4,-5199.793478,-5714.654702,-2245.324442,-7669.351393,-688.032158,-4303.431235,2507.243487,6
0,0.145133,0.054151,0.007668,0.005128,2,1,"{'knr3__max_depth': 2, 'knr3__min_samples_leaf...",-23.169788,-32.659638,-16.008907,...,6.802928,5,-6477.864861,-6764.323877,-917.611503,-7327.234063,-805.360124,-4458.478886,2949.834423,7
6,0.189600,0.004531,0.005170,0.003614,5,1,"{'knr3__max_depth': 5, 'knr3__min_samples_leaf...",-23.662884,-28.504918,-18.465169,...,6.126632,6,-6165.942045,-7181.155529,-1055.929361,-7972.272963,-722.747575,-4619.609495,3100.914823,8


In [56]:
Best_param3 = grid_cv3.best_params_
#Best_param1 = pd.DataFrame(Best_param1)
print(Best_param3)
print(grid_cv3.best_estimator_)

{'knr3__max_depth': 2, 'knr3__min_samples_leaf': 2}
Pipeline(steps=[('preprocess3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('knr3',
                 GradientBoostingRegressor(max_depth=2, min_samples_leaf=2))])


In [57]:
param_grid4 = {
    'knr4__max_depth': [2, 3, 4,5],
    'knr4__min_samples_leaf': [1,2]
    }

grid_cv4 = GridSearchCV(
    estimator=pipe_knr4, 
    param_grid=param_grid4, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_mean_squared_error")

grid_cv4.fit(X_train, Y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess4',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numeric_transfomer',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('Power_transfomer',
                                                                         FunctionTransformer(),
                                                                         ['temp']),
                                                                        ('onehot',
                                                                         OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('knr4', GradientBoostingRegressor())]),
             param_grid={'knr4__max_depth': [2, 3, 4, 5],
                         'knr4__min_samples_leaf': [1, 2]},
             refit='neg_mean_squared_error',
             scoring=['neg_mean_absolute_error', 'neg_mean_squared_error'])

In [58]:
res4 = grid_cv4.cv_results_
res4 = pd.DataFrame(res4)
res4
#res4.columns

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr4__max_depth,param_knr4__min_samples_leaf,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
0,0.143845,0.033553,0.009681,0.006256,2,1,"{'knr4__max_depth': 2, 'knr4__min_samples_leaf...",-23.192839,-29.009950,-15.976775,...,5.652576,3,-6486.591055,-4891.363611,-914.975751,-7337.747227,-807.942152,-4087.723959,2749.048969,5
1,0.128053,0.008777,0.004966,0.004700,2,2,"{'knr4__max_depth': 2, 'knr4__min_samples_leaf...",-20.118679,-26.960019,-15.624935,...,5.266486,1,-2450.342724,-3231.795693,-955.363530,-7480.499626,-657.737662,-2955.147847,2453.055313,1
2,0.141476,0.007480,0.010136,0.002660,3,1,"{'knr4__max_depth': 3, 'knr4__min_samples_leaf...",-21.873374,-28.219209,-23.743714,...,5.183871,8,-5156.977557,-6061.486889,-2766.864498,-7685.898193,-928.743813,-4519.994190,2398.976394,8
3,0.146544,0.002800,0.007974,0.004178,3,2,"{'knr4__max_depth': 3, 'knr4__min_samples_leaf...",-22.155750,-26.224123,-19.540232,...,4.809516,2,-3066.608147,-2872.974778,-1283.694042,-7400.169320,-759.257088,-3076.540675,2337.494585,2
4,0.183308,0.010217,0.006465,0.004159,4,1,"{'knr4__max_depth': 4, 'knr4__min_samples_leaf...",-22.451325,-29.880947,-16.790499,...,6.515893,6,-4524.165429,-6630.169578,-1022.621217,-8135.245795,-778.445819,-4218.129568,2942.722971,6
5,0.180267,0.013224,0.009129,0.004213,4,2,"{'knr4__max_depth': 4, 'knr4__min_samples_leaf...",-22.555787,-28.170362,-19.549831,...,4.868962,4,-3555.301671,-3135.570798,-1236.060478,-7205.493004,-847.204490,-3195.926088,2261.282237,4
6,0.211168,0.007403,0.007756,0.003259,5,1,"{'knr4__max_depth': 5, 'knr4__min_samples_leaf...",-23.581715,-27.376229,-19.966743,...,5.152053,7,-6014.287879,-6136.839711,-1213.040588,-7754.903729,-849.641374,-4393.742656,2815.651877,7
7,0.210687,0.008365,0.006740,0.003609,5,2,"{'knr4__max_depth': 5, 'knr4__min_samples_leaf...",-21.624356,-29.863510,-20.786820,...,5.235800,5,-2998.566018,-2952.431677,-1264.034864,-7353.303004,-859.327247,-3085.532562,2302.736960,3


In [59]:
res4.columns

Index(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_knr4__max_depth', 'param_knr4__min_samples_leaf', 'params',
       'split0_test_neg_mean_absolute_error',
       'split1_test_neg_mean_absolute_error',
       'split2_test_neg_mean_absolute_error',
       'split3_test_neg_mean_absolute_error',
       'split4_test_neg_mean_absolute_error',
       'mean_test_neg_mean_absolute_error', 'std_test_neg_mean_absolute_error',
       'rank_test_neg_mean_absolute_error',
       'split0_test_neg_mean_squared_error',
       'split1_test_neg_mean_squared_error',
       'split2_test_neg_mean_squared_error',
       'split3_test_neg_mean_squared_error',
       'split4_test_neg_mean_squared_error',
       'mean_test_neg_mean_squared_error', 'std_test_neg_mean_squared_error',
       'rank_test_neg_mean_squared_error'],
      dtype='object')

In [ ]:
res4[['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_knr4__max_depth', 'param_knr4__min_samples_leaf', 'params',
       'split0_test_neg_mean_absolute_error',
       'split1_test_neg_mean_absolute_error',
       'split2_test_neg_mean_absolute_error',
       'split3_test_neg_mean_absolute_error',
       'split4_test_neg_mean_absolute_error',
       'mean_test_neg_mean_absolute_error', 'std_test_neg_mean_absolute_error',
       'rank_test_neg_mean_absolute_error',
       'split0_test_neg_mean_squared_error',
       'split1_test_neg_mean_squared_error',
       'split2_test_neg_mean_squared_error',
       'split3_test_neg_mean_squared_error',
       'split4_test_neg_mean_squared_error',
       'mean_test_neg_mean_squared_error', 'std_test_neg_mean_squared_error',
       'rank_test_neg_mean_squared_error']].sort_values('rank_test_neg_mean_squared_error')

#Note that best param {'knr3__max_depth': 2, 'knr3__min_samples_leaf': 2}, avg mean = -2955.1

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr4__max_depth,param_knr4__min_samples_leaf,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
1,0.128053,0.008777,0.004966,0.004700,2,2,"{'knr4__max_depth': 2, 'knr4__min_samples_leaf...",-20.118679,-26.960019,-15.624935,...,5.266486,1,-2450.342724,-3231.795693,-955.363530,-7480.499626,-657.737662,-2955.147847,2453.055313,1
3,0.146544,0.002800,0.007974,0.004178,3,2,"{'knr4__max_depth': 3, 'knr4__min_samples_leaf...",-22.155750,-26.224123,-19.540232,...,4.809516,2,-3066.608147,-2872.974778,-1283.694042,-7400.169320,-759.257088,-3076.540675,2337.494585,2
7,0.210687,0.008365,0.006740,0.003609,5,2,"{'knr4__max_depth': 5, 'knr4__min_samples_leaf...",-21.624356,-29.863510,-20.786820,...,5.235800,5,-2998.566018,-2952.431677,-1264.034864,-7353.303004,-859.327247,-3085.532562,2302.736960,3
5,0.180267,0.013224,0.009129,0.004213,4,2,"{'knr4__max_depth': 4, 'knr4__min_samples_leaf...",-22.555787,-28.170362,-19.549831,...,4.868962,4,-3555.301671,-3135.570798,-1236.060478,-7205.493004,-847.204490,-3195.926088,2261.282237,4
0,0.143845,0.033553,0.009681,0.006256,2,1,"{'knr4__max_depth': 2, 'knr4__min_samples_leaf...",-23.192839,-29.009950,-15.976775,...,5.652576,3,-6486.591055,-4891.363611,-914.975751,-7337.747227,-807.942152,-4087.723959,2749.048969,5
4,0.183308,0.010217,0.006465,0.004159,4,1,"{'knr4__max_depth': 4, 'knr4__min_samples_leaf...",-22.451325,-29.880947,-16.790499,...,6.515893,6,-4524.165429,-6630.169578,-1022.621217,-8135.245795,-778.445819,-4218.129568,2942.722971,6
6,0.211168,0.007403,0.007756,0.003259,5,1,"{'knr4__max_depth': 5, 'knr4__min_samples_leaf...",-23.581715,-27.376229,-19.966743,...,5.152053,7,-6014.287879,-6136.839711,-1213.040588,-7754.903729,-849.641374,-4393.742656,2815.651877,7
2,0.141476,0.007480,0.010136,0.002660,3,1,"{'knr4__max_depth': 3, 'knr4__min_samples_leaf...",-21.873374,-28.219209,-23.743714,...,5.183871,8,-5156.977557,-6061.486889,-2766.864498,-7685.898193,-928.743813,-4519.994190,2398.976394,8


# Evaluate

+ Which model has the best performance?

** Pipeline A = preproc1 + baseline has the best mean_test_neg_mean_squared_error.

# Export

+ Save the best performing model to a pickle file.

In [61]:
import pickle 
filename = 'model_pipe_knr.pkl'
with open(filename, 'wb') as file:
        pickle.dump(pipe_knr, file)

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [62]:
import shap

In [70]:
# Not able to get Shap to work with KNeighborsRegressor
#import shap
pipe_knr.fit(X_train, Y_train)
data_transform = grid_cv1.best_estimator_.named_steps['preprocess'].transform(X_test)

explainer = shap.Explainer(
    grid_cv1.best_estimator_.named_steps['knr'], 
    data_transform,
    feature_names = grid_cv1.best_estimator_.named_steps['preprocess'].get_feature_names_out())

shap_values = explainer(data_transform)


TypeError: The passed model is not callable and cannot be analyzed directly with the given masker! Model: KNeighborsRegressor(n_neighbors=9)

In [71]:
# No luck with GradientBoostingRegressor either...
pipe_knr3.fit(X_train, Y_train)
data_transform = grid_cv3.best_estimator_.named_steps['preprocess3'].transform(X_test)

explainer = shap.Explainer(
    grid_cv3.best_estimator_.named_steps['knr3'], 
    data_transform,
    feature_names = grid_cv3.best_estimator_.named_steps['preprocess3'].get_feature_names_out())

shap_values = explainer(data_transform)


ExplainerError: Additivity check failed in TreeExplainer! Please ensure the data matrix you passed to the explainer is the same shape that the model was trained on. If your data shape is correct then please report this on GitHub. This check failed because for one of the samples the sum of the SHAP values was 71.407151, while the model output was 69.457846. If this difference is acceptable you can set check_additivity=False to disable this check.

In [ ]:
#shap.plots.waterfall(shap_values[1])

In [ ]:
#shap.plots.beeswarm(shap_values)

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.